# Delhivery: Extra Trees, Rotation Forest, Bagging, and Boosting Ensemble — v1

**A version that reuses the data-splitting protocol of the original notebook**

- ETA Prediction: Extra Trees, Rotation Forest, Bagging, and Boosting Ensemble.
- Severe Delay Classification: Extra Trees, Rotation Forest, Bagging, and Boosting Ensemble.
- Delay Propagation: Extra Trees Regressor only.
- The external training set is split sequentially into 80% fit and 20% validation, as in the original notebook.
- The entire validation set is used to evaluate and choose among the 4 candidates.
- The boosting meta-model only learns from time-ordered out-of-fold predictions inside the fit set, so there is no need to split validation into meta-train/meta-evaluation.


## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Data Loading](#2-data-loading)
3. [Data Quality Assessment](#3-data-quality-assessment)
4. [Preprocessing](#4-preprocessing)
5. [Aggregation: Segment → Leg → Trip](#5-aggregation-segment--leg--trip)
6. [EDA](#6-eda)
7. [Feature Engineering](#7-feature-engineering)
8. [Data Splitting and Leakage Prevention](#8-data-splitting-and-leakage-prevention)
9. [ETA Prediction — 3 Base Models + Boosting Ensemble](#9-eta-prediction--3-base-models--boosting-ensemble)
10. [Delay Magnitude Model](#10-delay-magnitude-model)
11. [Severe Delay Classification — 3 Base Models + Boosting Ensemble](#11-severe-delay-classification--3-base-models--boosting-ensemble)
12. [Delay Propagation — Extra Trees Regressor Only](#12-delay-propagation--extra-trees-regressor-only)
13. [Explainability with LIME/SHAP — Temporarily Commented Out](#13-explainability-with-limeshap--temporarily-commented-out)
14. [Single-Shipment Prediction Example](#14-single-shipment-prediction-example)
15. [Model Export and Conclusion](#15-model-export-and-conclusion)


## 1. Environment Setup


This notebook uses only `numpy`, `pandas`, `scipy`, `scikit-learn`, `matplotlib`, `seaborn`, and `joblib`. XGBoost/LightGBM/CatBoost/imbalanced-learn are no longer installed or imported, since those models are not used.


In [ ]:
AUTO_INSTALL_NEW_PACKAGES = False
print('Không cài dependency bổ sung: chỉ dùng scikit-learn cho toàn bộ model.')

In [ ]:
from pathlib import Path
import os

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')

import gc
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

from IPython.display import display

from sklearn.base import BaseEstimator, RegressorMixin, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (
    ExtraTreesRegressor,
    ExtraTreesClassifier,
    BaggingRegressor,
    BaggingClassifier,
    GradientBoostingRegressor,
    GradientBoostingClassifier,
)
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    root_mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
DELAY_THRESHOLD_MINUTES = 120
N_JOBS = 2

print('Model scope: 3 base models + 1 boosting ensemble cho ETA/classification; Extra Trees cho propagation.')

## 2. Data Loading


On Kaggle, the dataset is loaded directly from the `/kaggle/input` path. The cell below prioritizes the user-supplied path and falls back to local paths when running locally.


In [ ]:
KAGGLE_DATA_PATH = Path(
    '/kaggle/input/datasets/santanukundu/delhivery-dataset/delhivery_data.csv'
)


def find_data_path(filename='delhivery_data.csv'):
    candidates = [
        KAGGLE_DATA_PATH,
        Path('/kaggle/input/delhivery-dataset') / filename,
        Path(filename),
        Path('data') / filename,
        Path('Delhivery-Logistics-Shipment-Data-Analysis-main') / filename,
        Path('repo2') / 'Delhivery-Logistics-Shipment-Data-Analysis-main' / filename,
    ]

    for path in candidates:
        if path.exists():
            return path.resolve()

    matches = list(Path.cwd().glob(f'**/{filename}'))
    if matches:
        return matches[0].resolve()

    checked = '\n'.join(f'- {path}' for path in candidates)
    raise FileNotFoundError(
        f'Không tìm thấy {filename}. Các đường dẫn đã kiểm tra:\n{checked}'
    )


DATA_PATH = find_data_path()
print(f'Data path: {DATA_PATH}')

raw_df = pd.read_csv(DATA_PATH)
print(f'Raw shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns')
display(raw_df.head())


### Condensed Data Dictionary

| Category | Main Fields | Purpose |
|---|---|---|
| ID | `trip_uuid`, `route_schedule_uuid` | Identify the trip and route schedule |
| Route | `source_*`, `destination_*`, `route_type` | Geographic and transport-type features |
| Ground truth | `od_start_time`, `od_end_time` | Compute the actual duration and final ETA |
| Plan | `osrm_time`, `osrm_distance`, `segment_osrm_*` | Routing baseline used to compute the proxy delay |
| Dynamic actuals | `segment_actual_time`, `cutoff_timestamp` | Only usable after the checkpoint has occurred |
| Audit | `actual_time`, `factor`, `segment_factor` | Not used as a start-of-trip target/feature because of leakage risk or because it does not represent the full elapsed time |


## 3. Data Quality Assessment


In [ ]:
quality_summary = pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing_count': raw_df.isna().sum(),
    'missing_pct': raw_df.isna().mean().mul(100),
    'n_unique': raw_df.nunique(dropna=False),
}).sort_values('missing_pct', ascending=False)

display(quality_summary)

print(f'Exact duplicate rows: {raw_df.duplicated().sum():,}')
print(f'Unique trips: {raw_df["trip_uuid"].nunique():,}')
print(f'Negative segment_actual_time rows: {(raw_df["segment_actual_time"] < 0).sum():,}')
print(f'Non-positive segment_osrm_time rows: {(raw_df["segment_osrm_time"] <= 0).sum():,}')

In [ ]:
missing = quality_summary.query('missing_count > 0').sort_values('missing_pct')
if not missing.empty:
    plt.figure(figsize=(9, 4))
    sns.barplot(data=missing.reset_index(), x='missing_pct', y='index', color='C0')
    plt.xlabel('Missing (%)')
    plt.ylabel('Column')
    plt.title('Tỷ lệ missing value')
    plt.tight_layout()
    plt.show()
else:
    print('Không có missing value.')

## 4. Preprocessing


Processing principles:

- Do not modify the original dataframe directly; work on a copy.
- Convert columns to the correct datetime type.
- Fill missing location names with the most common/first name within the same `center`, then fall back to the center code.
- Split `city` and `state` out of the location name.
- Keep anomaly flags instead of removing all logistics outliers, since long trips or large delays may be genuine data.


In [ ]:
df = raw_df.copy()
df = df.drop_duplicates().reset_index(drop=True)

DATETIME_COLUMNS = [
    'trip_creation_time', 'od_start_time', 'od_end_time', 'cutoff_timestamp'
]


def parse_mixed_datetime(series):
    """Parse timestamp có/không có microseconds mà không làm mất dữ liệu hợp lệ."""
    try:
        return pd.to_datetime(series, format='mixed', errors='coerce')
    except TypeError:
        return pd.to_datetime(series, errors='coerce')


for col in DATETIME_COLUMNS:
    df[col] = parse_mixed_datetime(df[col])

df['source_name'] = (
    df['source_name']
      .fillna(df.groupby('source_center')['source_name'].transform('first'))
      .fillna(df['source_center'])
)
df['destination_name'] = (
    df['destination_name']
      .fillna(df.groupby('destination_center')['destination_name'].transform('first'))
      .fillna(df['destination_center'])
)


def extract_city(series):
    return series.astype(str).str.split('_').str[0].str.strip()


def extract_state(series):
    return (
        series.astype(str)
              .str.extract(r'\(([^()]*)\)\s*$', expand=False)
              .fillna('Unknown')
              .str.strip()
    )


df['source_city'] = extract_city(df['source_name'])
df['destination_city'] = extract_city(df['destination_name'])
df['source_state'] = extract_state(df['source_name'])
df['destination_state'] = extract_state(df['destination_name'])

df['negative_segment_time_flag'] = (df['segment_actual_time'] < 0).astype(int)
df['zero_segment_plan_flag'] = (df['segment_osrm_time'] <= 0).astype(int)
df['_row_order'] = np.arange(len(df))

print('Remaining missing values:')
display(df.isna().sum()[df.isna().sum() > 0])
print(f"Missing cutoff_timestamp after mixed parsing: {df['cutoff_timestamp'].isna().sum():,}")
display(df.head())

## 5. Aggregation: Segment → Leg → Trip


The raw dataset has multiple checkpoints for the same leg and multiple legs for the same trip. Training directly on individual rows could let checkpoints from the same trip fall into both train and test, artificially inflating the metric.

Aggregation pipeline:

1. **Checkpoint row**: a dynamic observation recorded during the journey.
2. **Leg**: rows sharing `trip_uuid + source + destination + od_start_time`.
3. **Trip**: a single row for each `trip_uuid`.

Checkpoints are sorted by `cutoff_timestamp`. Cumulative fields at the leg level use `max`; incremental fields use `sum`. The final ground truth is not taken from `actual_time`, but is computed from the trip's start/end timestamps.


In [ ]:
df_sorted = df.sort_values(
    ['trip_uuid', 'od_start_time', 'cutoff_timestamp', '_row_order'],
    na_position='last',
).copy()

leg_keys = ['trip_uuid', 'source_center', 'destination_center', 'od_start_time']

leg_df = df_sorted.groupby(leg_keys, as_index=False).agg(
    data=('data', 'first'),
    route_schedule_uuid=('route_schedule_uuid', 'first'),
    route_type=('route_type', 'first'),
    trip_creation_time=('trip_creation_time', 'first'),
    source_name=('source_name', 'first'),
    destination_name=('destination_name', 'last'),
    source_city=('source_city', 'first'),
    destination_city=('destination_city', 'last'),
    source_state=('source_state', 'first'),
    destination_state=('destination_state', 'last'),
    od_end_time=('od_end_time', 'max'),
    start_scan_to_end_scan=('start_scan_to_end_scan', 'max'),
    actual_distance_to_destination=('actual_distance_to_destination', 'max'),
    actual_time=('actual_time', 'max'),
    osrm_time=('osrm_time', 'max'),
    osrm_distance=('osrm_distance', 'max'),
    segment_actual_time=('segment_actual_time', 'sum'),
    segment_osrm_time=('segment_osrm_time', 'sum'),
    segment_osrm_distance=('segment_osrm_distance', 'sum'),
    checkpoint_count=('trip_uuid', 'size'),
    cutoff_rate=('is_cutoff', 'mean'),
    anomaly_count=('negative_segment_time_flag', 'sum'),
    zero_plan_count=('zero_segment_plan_flag', 'sum'),
)

leg_df = leg_df.sort_values(['trip_uuid', 'od_start_time']).reset_index(drop=True)
print(f'Leg-level shape: {leg_df.shape}')
display(leg_df.head())

In [ ]:
trip_df = leg_df.groupby('trip_uuid', as_index=False).agg(
    data=('data', 'first'),
    route_schedule_uuid=('route_schedule_uuid', 'first'),
    route_type=('route_type', 'first'),
    trip_creation_time=('trip_creation_time', 'first'),
    source_center=('source_center', 'first'),
    destination_center=('destination_center', 'last'),
    source_name=('source_name', 'first'),
    destination_name=('destination_name', 'last'),
    source_city=('source_city', 'first'),
    destination_city=('destination_city', 'last'),
    source_state=('source_state', 'first'),
    destination_state=('destination_state', 'last'),
    trip_start_time=('od_start_time', 'min'),
    trip_end_time=('od_end_time', 'max'),
    leg_count=('trip_uuid', 'size'),
    checkpoint_count=('checkpoint_count', 'sum'),
    start_scan_to_end_scan=('start_scan_to_end_scan', 'sum'),
    actual_distance_to_destination=('actual_distance_to_destination', 'sum'),
    actual_time=('actual_time', 'sum'),
    osrm_time=('osrm_time', 'sum'),
    osrm_distance=('osrm_distance', 'sum'),
    segment_actual_time=('segment_actual_time', 'sum'),
    segment_osrm_time=('segment_osrm_time', 'sum'),
    segment_osrm_distance=('segment_osrm_distance', 'sum'),
    cutoff_rate=('cutoff_rate', 'mean'),
    anomaly_count=('anomaly_count', 'sum'),
    zero_plan_count=('zero_plan_count', 'sum'),
)

trip_df['trip_duration_minutes'] = (
    trip_df['trip_end_time'] - trip_df['trip_start_time']
).dt.total_seconds() / 60

trip_df = trip_df.query(
    'trip_duration_minutes > 0 and osrm_time > 0 and osrm_distance > 0'
).copy()
trip_df = trip_df.sort_values('trip_creation_time').reset_index(drop=True)

print(f'Trip-level shape: {trip_df.shape}')
print(f'Unique trip_uuid after aggregation: {trip_df.trip_uuid.nunique():,}')
display(trip_df.head())

In [ ]:
trip_df['actual_time_gap_vs_elapsed'] = (
    trip_df['trip_duration_minutes'] - trip_df['actual_time']
)

consistency = pd.DataFrame({
    'metric': [
        'trip_duration vs actual_time (audit only)',
        'actual_time vs segment_actual_time',
        'osrm_time vs segment_osrm_time',
        'osrm_distance vs segment_osrm_distance',
    ],
    'correlation': [
        trip_df['trip_duration_minutes'].corr(trip_df['actual_time']),
        trip_df['actual_time'].corr(trip_df['segment_actual_time']),
        trip_df['osrm_time'].corr(trip_df['segment_osrm_time']),
        trip_df['osrm_distance'].corr(trip_df['segment_osrm_distance']),
    ]
})
display(consistency)

display(
    trip_df['actual_time_gap_vs_elapsed']
           .describe(percentiles=[.25, .5, .75, .9, .95, .99])
           .to_frame('elapsed_minus_actual_time_minutes')
)

## 6. EDA


In [ ]:
trip_df['actual_eta'] = trip_df['trip_end_time']
trip_df['osrm_eta'] = trip_df['trip_start_time'] + pd.to_timedelta(trip_df['osrm_time'], unit='m')
trip_df['delay_minutes'] = trip_df['trip_duration_minutes'] - trip_df['osrm_time']
trip_df['delay_ratio'] = trip_df['trip_duration_minutes'] / trip_df['osrm_time']

summary_cols = [
    'trip_duration_minutes', 'actual_time', 'osrm_time', 'delay_minutes',
    'osrm_distance', 'leg_count', 'checkpoint_count'
]
display(trip_df[summary_cols].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]).T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = trip_df.sample(min(5000, len(trip_df)), random_state=RANDOM_STATE)
sns.scatterplot(
    data=sample, x='osrm_time', y='trip_duration_minutes',
    hue='route_type', alpha=.55, ax=axes[0]
)
max_time = max(sample['osrm_time'].max(), sample['trip_duration_minutes'].max())
axes[0].plot([0, max_time], [0, max_time], linestyle='--', linewidth=1)
axes[0].set_title('Full elapsed duration so với OSRM baseline')
axes[0].set_xlabel('OSRM time (phút)')
axes[0].set_ylabel('Trip duration thực (phút)')

sns.histplot(trip_df['delay_minutes'], bins=80, kde=True, ax=axes[1], color='C0')
axes[1].axvline(
    DELAY_THRESHOLD_MINUTES, linestyle='--',
    label=f'Threshold = {DELAY_THRESHOLD_MINUTES} phút'
)
axes[1].set_xlim(trip_df['delay_minutes'].quantile(.01), trip_df['delay_minutes'].quantile(.99))
axes[1].set_title('Phân phối proxy delay (cắt 1% hai đầu khi hiển thị)')
axes[1].set_xlabel('Delay = full elapsed duration - OSRM time (phút)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=trip_df, x='route_type', ax=axes[0], color='C0')
axes[0].set_title('Số trip theo route type')
axes[0].set_xlabel('Route type')
axes[0].set_ylabel('Số trip')

route_delay = (
    trip_df.groupby('route_type', as_index=False)
           .agg(mean_delay=('delay_minutes', 'mean'), median_delay=('delay_minutes', 'median'))
)
route_delay_melt = route_delay.melt('route_type', var_name='metric', value_name='delay_minutes')
sns.barplot(data=route_delay_melt, x='route_type', y='delay_minutes', hue='metric', ax=axes[1], palette='deep')
axes[1].set_title('Delay theo route type')
axes[1].set_xlabel('Route type')
axes[1].set_ylabel('Phút')

plt.tight_layout()
plt.show()

In [ ]:
trip_df['state_corridor'] = trip_df['source_state'] + ' → ' + trip_df['destination_state']

corridor_summary = (
    trip_df.groupby('state_corridor')
           .agg(
               trips=('trip_uuid', 'size'),
               mean_delay=('delay_minutes', 'mean'),
               median_delay=('delay_minutes', 'median'),
               mean_trip_duration=('trip_duration_minutes', 'mean'),
           )
           .query('trips >= 20')
           .sort_values('mean_delay', ascending=False)
           .head(15)
           .reset_index()
)

display(corridor_summary)

plt.figure(figsize=(11, 6))
sns.barplot(data=corridor_summary, x='mean_delay', y='state_corridor', color='C0')
plt.title('Top corridor có mean proxy delay cao (tối thiểu 20 trips)')
plt.xlabel('Mean delay (phút)')
plt.ylabel('State corridor')
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = [
    'trip_duration_minutes', 'actual_time', 'osrm_time', 'delay_minutes',
    'osrm_distance', 'segment_actual_time', 'segment_osrm_time',
    'segment_osrm_distance', 'leg_count', 'checkpoint_count'
]
plt.figure(figsize=(11, 8))
sns.heatmap(trip_df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation tại cấp trip')
plt.tight_layout()
plt.show()

## 7. Feature Engineering


### Features for the Start-of-Trip Model

Only information guaranteed to be available before or at departure is used:

- the route plan's `osrm_time`, `osrm_distance`;
- origin, destination, and route type;
- departure hour/day;
- ratios computed only from the OSRM plan.

`segment_osrm_*`, `leg_count`, and `checkpoint_count` are not used because they are aggregated from all actual rows and may not be fully known at departure. No actual/cutoff/end-time variable is used.


In [ ]:
trip_df['start_hour'] = trip_df['trip_start_time'].dt.hour
trip_df['start_dayofweek'] = trip_df['trip_start_time'].dt.dayofweek
trip_df['start_month'] = trip_df['trip_start_time'].dt.month
trip_df['is_weekend'] = trip_df['start_dayofweek'].isin([5, 6]).astype(int)
trip_df['state_corridor'] = trip_df['source_state'] + ' → ' + trip_df['destination_state']

trip_df['planned_speed_kmph'] = (
    trip_df['osrm_distance'] / (trip_df['osrm_time'] / 60).replace(0, np.nan)
)
trip_df['planned_minutes_per_km'] = (
    trip_df['osrm_time'] / trip_df['osrm_distance'].replace(0, np.nan)
)
trip_df = trip_df.replace([np.inf, -np.inf], np.nan)

START_NUMERIC_FEATURES = [
    'osrm_time', 'osrm_distance',
    'start_hour', 'start_dayofweek', 'start_month', 'is_weekend',
    'planned_speed_kmph', 'planned_minutes_per_km',
]

START_CATEGORICAL_FEATURES = [
    'route_type', 'source_state', 'destination_state', 'state_corridor'
]

START_FEATURES = START_NUMERIC_FEATURES + START_CATEGORICAL_FEATURES

print(f'Number of start-of-trip model features: {len(START_FEATURES)}')
display(trip_df[START_FEATURES + ['trip_duration_minutes', 'delay_minutes']].head())

## 8. Data Splitting and Leakage Prevention


The dataset has a built-in external time-based split:

- `training`: the earlier period;
- `test`: the later period.

The final test set is kept intact and used **only once**, after the model has been selected. Within `training`, the notebook creates an additional chronological 80/20 fit/validation split to compare models and choose the classification threshold.


In [ ]:
train_df = trip_df.query("data == 'training'").copy().sort_values('trip_creation_time')
test_df = trip_df.query("data == 'test'").copy().sort_values('trip_creation_time')

trip_overlap = set(train_df['trip_uuid']).intersection(test_df['trip_uuid'])
assert len(trip_overlap) == 0, 'Có trip_uuid trùng giữa external train và test.'

validation_start = int(len(train_df) * 0.80)
fit_df = train_df.iloc[:validation_start].copy()
validation_df = train_df.iloc[validation_start:].copy()

fit_val_overlap = set(fit_df['trip_uuid']).intersection(validation_df['trip_uuid'])
assert len(fit_val_overlap) == 0, 'Có trip_uuid trùng giữa fit và validation.'
assert fit_df['trip_creation_time'].max() <= validation_df['trip_creation_time'].min()

split_summary = pd.DataFrame({
    'split': ['fit', 'validation', 'final test'],
    'n_trips': [len(fit_df), len(validation_df), len(test_df)],
    'pct_all_trips': [len(fit_df)/len(trip_df), len(validation_df)/len(trip_df), len(test_df)/len(trip_df)],
    'start_date': [
        fit_df['trip_creation_time'].min(), validation_df['trip_creation_time'].min(),
        test_df['trip_creation_time'].min()
    ],
    'end_date': [
        fit_df['trip_creation_time'].max(), validation_df['trip_creation_time'].max(),
        test_df['trip_creation_time'].max()
    ],
    'mean_duration': [
        fit_df['trip_duration_minutes'].mean(), validation_df['trip_duration_minutes'].mean(),
        test_df['trip_duration_minutes'].mean()
    ],
    'mean_delay': [fit_df['delay_minutes'].mean(), validation_df['delay_minutes'].mean(), test_df['delay_minutes'].mean()],
})

display(split_summary)
print(f'External train trips: {len(train_df):,} ({len(train_df)/len(trip_df):.2%})')
print(f'Final test trips: {len(test_df):,} ({len(test_df)/len(trip_df):.2%})')
print(f'Trip overlap external train/test: {len(trip_overlap)}')

### Leakage Guard

The following columns **must not be fed into the start-of-trip ETA model**:

- `actual_time`, `segment_actual_time`, `actual_distance_to_destination`;
- `factor`, `segment_factor`, because they are derived directly from actual/planned time;
- `od_end_time`, `trip_end_time`, `start_scan_to_end_scan`;
- `cutoff_timestamp`, `cutoff_factor`, `cutoff_rate`, if predicting before the trip starts;
- `delay_minutes`, `actual_eta`, because they are the target or derived from the target.


In [ ]:
LEAKAGE_COLUMNS = {
    'actual_time', 'segment_actual_time', 'actual_distance_to_destination',
    'factor', 'segment_factor', 'od_end_time', 'trip_end_time',
    'start_scan_to_end_scan', 'cutoff_timestamp', 'cutoff_factor',
    'cutoff_rate', 'delay_minutes', 'delay_ratio', 'actual_eta',
    'trip_duration_minutes', 'actual_time_gap_vs_elapsed',
    'segment_osrm_time', 'segment_osrm_distance', 'leg_count', 'checkpoint_count'
}

leaked = LEAKAGE_COLUMNS.intersection(START_FEATURES)
assert not leaked, f'Feature set chứa biến leakage/future information: {leaked}'
print('Start-of-trip leakage check passed.')

In [ ]:
def make_start_preprocessor():
    return ColumnTransformer(
        transformers=[
            ('numeric', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]), START_NUMERIC_FEATURES),
            ('categorical', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(
                    handle_unknown='ignore',
                    min_frequency=10,
                    sparse_output=True,
                )),
            ]), START_CATEGORICAL_FEATURES),
        ],
        remainder='drop',
    )


def to_dense(X):
    return X.toarray() if sparse.issparse(X) else np.asarray(X, dtype=float)


def safe_roc_auc(y_true, probability):
    return roc_auc_score(y_true, probability) if pd.Series(y_true).nunique() > 1 else np.nan


def positive_class_probability(model, X):
    probability = np.asarray(model.predict_proba(X), dtype=float)
    if probability.ndim == 1:
        return probability.reshape(-1)
    classes = np.asarray(getattr(model, 'classes_', [0, 1]))
    positive_positions = np.flatnonzero(classes == 1)
    positive_position = int(positive_positions[0]) if len(positive_positions) else probability.shape[1] - 1
    return probability[:, positive_position]


def fit_classifier_with_balancing(model, X, y):
    sample_weight = compute_sample_weight(class_weight='balanced', y=y)
    try:
        model.fit(X, y, sample_weight=sample_weight)
        return model, True
    except (TypeError, ValueError):
        model.fit(X, y)
        return model, False


class RotationForestRegressor(BaseEstimator, RegressorMixin):
    """Rotation Forest: PCA theo nhóm feature, sau đó trung bình nhiều Decision Tree."""
    def __init__(self, n_estimators=30, n_feature_subsets=5, sample_fraction=0.75,
                 min_samples_leaf=2, random_state=None):
        self.n_estimators = n_estimators
        self.n_feature_subsets = n_feature_subsets
        self.sample_fraction = sample_fraction
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state

    @staticmethod
    def _rotate(X, rotations):
        X_rot = np.empty_like(X, dtype=float)
        for indices, pca in rotations:
            X_rot[:, indices] = pca.transform(X[:, indices])
        return X_rot

    def fit(self, X, y):
        X = to_dense(X)
        y = np.asarray(y)
        rng = np.random.RandomState(self.random_state)
        self.estimators_ = []
        self.rotations_ = []
        self.n_features_in_ = X.shape[1]
        subset_count = max(1, min(self.n_feature_subsets, X.shape[1]))
        sample_size = min(len(y), max(X.shape[1] + 1, int(self.sample_fraction * len(y))))

        for _ in range(self.n_estimators):
            groups = np.array_split(rng.permutation(X.shape[1]), subset_count)
            rotations = []
            X_rot = np.empty_like(X, dtype=float)
            for indices in groups:
                rows = rng.choice(len(y), size=sample_size, replace=False)
                pca = PCA(n_components=len(indices), random_state=rng.randint(0, 2**31 - 1))
                pca.fit(X[np.ix_(rows, indices)])
                X_rot[:, indices] = pca.transform(X[:, indices])
                rotations.append((indices, pca))
            tree = DecisionTreeRegressor(
                min_samples_leaf=self.min_samples_leaf,
                random_state=rng.randint(0, 2**31 - 1),
            ).fit(X_rot, y)
            self.rotations_.append(rotations)
            self.estimators_.append(tree)
        return self

    def predict(self, X):
        X = to_dense(X)
        predictions = [
            tree.predict(self._rotate(X, rotations))
            for tree, rotations in zip(self.estimators_, self.rotations_)
        ]
        return np.mean(predictions, axis=0)


class RotationForestClassifier(BaseEstimator, ClassifierMixin):
    """Rotation Forest classifier với xác suất là trung bình của các cây."""
    def __init__(self, n_estimators=30, n_feature_subsets=5, sample_fraction=0.75,
                 min_samples_leaf=2, random_state=None):
        self.n_estimators = n_estimators
        self.n_feature_subsets = n_feature_subsets
        self.sample_fraction = sample_fraction
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state

    @staticmethod
    def _rotate(X, rotations):
        X_rot = np.empty_like(X, dtype=float)
        for indices, pca in rotations:
            X_rot[:, indices] = pca.transform(X[:, indices])
        return X_rot

    def fit(self, X, y, sample_weight=None):
        X = to_dense(X)
        y = np.asarray(y)
        rng = np.random.RandomState(self.random_state)
        self.classes_ = np.unique(y)
        self.estimators_ = []
        self.rotations_ = []
        self.n_features_in_ = X.shape[1]
        subset_count = max(1, min(self.n_feature_subsets, X.shape[1]))
        sample_size = min(len(y), max(X.shape[1] + 1, int(self.sample_fraction * len(y))))

        for _ in range(self.n_estimators):
            groups = np.array_split(rng.permutation(X.shape[1]), subset_count)
            rotations = []
            X_rot = np.empty_like(X, dtype=float)
            for indices in groups:
                rows = rng.choice(len(y), size=sample_size, replace=False)
                pca = PCA(n_components=len(indices), random_state=rng.randint(0, 2**31 - 1))
                pca.fit(X[np.ix_(rows, indices)])
                X_rot[:, indices] = pca.transform(X[:, indices])
                rotations.append((indices, pca))
            tree = DecisionTreeClassifier(
                min_samples_leaf=self.min_samples_leaf,
                random_state=rng.randint(0, 2**31 - 1),
            )
            tree.fit(X_rot, y, sample_weight=sample_weight)
            self.rotations_.append(rotations)
            self.estimators_.append(tree)
        return self

    def predict_proba(self, X):
        X = to_dense(X)
        probabilities = [
            tree.predict_proba(self._rotate(X, rotations))
            for tree, rotations in zip(self.estimators_, self.rotations_)
        ]
        return np.mean(probabilities, axis=0)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


def make_extra_trees_regressor():
    return ExtraTreesRegressor(
        n_estimators=180,
        max_depth=None,
        min_samples_leaf=2,
        max_features=1.0,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    )


def make_extra_trees_classifier():
    return ExtraTreesClassifier(
        n_estimators=180,
        max_depth=None,
        min_samples_leaf=2,
        max_features=1.0,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    )


def make_bagging_regressor():
    base = DecisionTreeRegressor(min_samples_leaf=2, random_state=RANDOM_STATE)
    kwargs = dict(
        n_estimators=120,
        max_samples=1.0,
        max_features=1.0,
        bootstrap=True,
        bootstrap_features=False,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    )
    try:
        return BaggingRegressor(estimator=base, **kwargs)
    except TypeError:
        return BaggingRegressor(base_estimator=base, **kwargs)


def make_bagging_classifier():
    base = DecisionTreeClassifier(min_samples_leaf=2, random_state=RANDOM_STATE)
    kwargs = dict(
        n_estimators=120,
        max_samples=1.0,
        max_features=1.0,
        bootstrap=True,
        bootstrap_features=False,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    )
    try:
        return BaggingClassifier(estimator=base, **kwargs)
    except TypeError:
        return BaggingClassifier(base_estimator=base, **kwargs)


def make_rotation_regressor():
    return RotationForestRegressor(
        n_estimators=30,
        n_feature_subsets=5,
        sample_fraction=0.75,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
    )


def make_rotation_classifier():
    return RotationForestClassifier(
        n_estimators=30,
        n_feature_subsets=5,
        sample_fraction=0.75,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
    )


ETA_BASE_FACTORIES = {
    'Extra Trees Regressor': make_extra_trees_regressor,
    'Rotation Forest Regressor': make_rotation_regressor,
    'Bagging Regressor': make_bagging_regressor,
}
CLASSIFIER_BASE_FACTORIES = {
    'Extra Trees Classifier': make_extra_trees_classifier,
    'Rotation Forest Classifier': make_rotation_classifier,
    'Bagging Classifier': make_bagging_classifier,
}
ETA_BASE_ORDER = list(ETA_BASE_FACTORIES)
CLASSIFIER_BASE_ORDER = list(CLASSIFIER_BASE_FACTORIES)
ETA_ENSEMBLE_NAME = 'Boosting Ensemble (3 regressors)'
CLASSIFIER_ENSEMBLE_NAME = 'Boosting Ensemble (3 classifiers)'


def make_eta_meta_model():
    return GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=2,
        min_samples_leaf=5,
        loss='huber',
        random_state=RANDOM_STATE,
    )


def make_classifier_meta_model():
    return GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=2,
        min_samples_leaf=5,
        subsample=0.90,
        random_state=RANDOM_STATE,
    )


class BoostingPredictionRegressor(BaseEstimator, RegressorMixin):
    """Dùng 3 prediction làm feature cho GradientBoostingRegressor."""
    def __init__(self, base_models, meta_model, model_order):
        self.base_models = base_models
        self.meta_model = meta_model
        self.model_order = model_order

    def predict(self, X):
        meta_X = np.column_stack([
            self.base_models[name].predict(X) for name in self.model_order
        ])
        return self.meta_model.predict(meta_X)


class BoostingPredictionClassifier(BaseEstimator, ClassifierMixin):
    """Dùng 3 xác suất severe-delay làm feature cho GradientBoostingClassifier."""
    def __init__(self, base_models, meta_model, model_order):
        self.base_models = base_models
        self.meta_model = meta_model
        self.model_order = model_order
        self.classes_ = np.array([0, 1])

    def predict_proba(self, X):
        meta_X = np.column_stack([
            positive_class_probability(self.base_models[name], X)
            for name in self.model_order
        ])
        positive = positive_class_probability(self.meta_model, meta_X)
        return np.column_stack([1.0 - positive, positive])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


model_inventory_df = pd.DataFrame([
    {'Task': 'ETA Prediction', 'Case': name, 'Role': 'Base model'}
    for name in ETA_BASE_ORDER
] + [
    {'Task': 'ETA Prediction', 'Case': ETA_ENSEMBLE_NAME, 'Role': 'Gradient Boosting meta-model'}
] + [
    {'Task': 'Severe Delay Classification', 'Case': name, 'Role': 'Base model'}
    for name in CLASSIFIER_BASE_ORDER
] + [
    {'Task': 'Severe Delay Classification', 'Case': CLASSIFIER_ENSEMBLE_NAME, 'Role': 'Gradient Boosting meta-model'}
] + [
    {'Task': 'Delay Propagation', 'Case': 'Extra Trees Regressor', 'Role': 'Only model'}
])
display(model_inventory_df.style.set_caption('Model inventory — Delhivery-Extra'))

## 9. ETA Prediction — 3 Base Models + Boosting Ensemble


Three base models are trained independently: **Extra Trees Regressor**, **Rotation Forest Regressor**, and **Bagging Regressor**.

The data split reverts to match the original notebook:

- the first 80% of external training is used as `fit_df`;
- the final 20% is kept intact as `validation_df`;
- all four candidates are evaluated on the complete validation set.

The ensemble does not use voting. Inside `fit_df`, `TimeSeriesSplit` generates time-ordered out-of-fold predictions for the three base models. These three predictions become the input to a `GradientBoostingRegressor`. Since the meta-model never learns from validation, validation remains an independent evaluation set.


In [ ]:
STACKING_N_SPLITS = 5
print(
    'Split protocol: 80% fit + 20% validation như notebook ban đầu; '
    f'boosting meta-model dùng {STACKING_N_SPLITS}-fold chronological OOF chỉ bên trong tập fit.'
)


In [ ]:
def regression_metrics(y_true, y_pred):
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0.0)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MedianAE': median_absolute_error(y_true, y_pred),
        'RMSE': root_mean_squared_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }


def fit_regressor_dict(factories, X, y):
    models = {}
    for position, (name, factory) in enumerate(factories.items(), start=1):
        start = time.perf_counter()
        model = factory()
        model.fit(X, y)
        models[name] = model
        print(f'[{position}/{len(factories)}] fitted {name}: {time.perf_counter() - start:.2f}s')
        gc.collect()
    return models


def chronological_oof_regression(raw_data, factories, target_column, n_splits=5):
    """Tạo prediction OOF theo thời gian; fit preprocessor riêng trong từng fold."""
    raw_data = raw_data.sort_values('trip_creation_time').reset_index(drop=True)
    y = raw_data[target_column].to_numpy(dtype=float)
    model_names = list(factories)
    oof_predictions = np.full((len(raw_data), len(model_names)), np.nan, dtype=float)
    fold_rows = []
    splitter = TimeSeriesSplit(n_splits=n_splits)

    for fold, (train_idx, holdout_idx) in enumerate(splitter.split(raw_data), start=1):
        fold_start = time.perf_counter()
        fold_preprocessor = make_start_preprocessor()
        X_fold_train = to_dense(
            fold_preprocessor.fit_transform(raw_data.iloc[train_idx][START_FEATURES])
        )
        X_fold_holdout = to_dense(
            fold_preprocessor.transform(raw_data.iloc[holdout_idx][START_FEATURES])
        )
        y_fold_train = y[train_idx]

        print(
            f'[ETA OOF fold {fold}/{n_splits}] '
            f'train={len(train_idx):,}, holdout={len(holdout_idx):,}'
        )
        for model_position, (name, factory) in enumerate(factories.items()):
            model_start = time.perf_counter()
            model = factory()
            model.fit(X_fold_train, y_fold_train)
            oof_predictions[holdout_idx, model_position] = np.maximum(
                np.asarray(model.predict(X_fold_holdout), dtype=float), 0.0
            )
            print(f'  - {name}: {time.perf_counter() - model_start:.2f}s')
            del model
            gc.collect()

        fold_rows.append({
            'Fold': fold,
            'Train rows': len(train_idx),
            'OOF rows': len(holdout_idx),
            'Train end': raw_data.iloc[train_idx]['trip_creation_time'].max(),
            'OOF start': raw_data.iloc[holdout_idx]['trip_creation_time'].min(),
            'OOF end': raw_data.iloc[holdout_idx]['trip_creation_time'].max(),
            'Seconds': time.perf_counter() - fold_start,
        })
        del fold_preprocessor, X_fold_train, X_fold_holdout
        gc.collect()

    valid_mask = np.isfinite(oof_predictions).all(axis=1)
    if not valid_mask.any():
        raise RuntimeError('Không tạo được prediction OOF cho ETA ensemble.')

    return (
        oof_predictions[valid_mask],
        y[valid_mask],
        valid_mask,
        pd.DataFrame(fold_rows),
    )


validation_preprocessor = make_start_preprocessor()
X_fit_t = to_dense(validation_preprocessor.fit_transform(fit_df[START_FEATURES]))
X_validation_t = to_dense(validation_preprocessor.transform(validation_df[START_FEATURES]))
y_fit_eta = fit_df['trip_duration_minutes'].to_numpy()
y_validation_eta = validation_df['trip_duration_minutes'].to_numpy()

eta_fit_oof_X, eta_fit_oof_y, eta_fit_oof_mask, eta_oof_fold_summary = (
    chronological_oof_regression(
        fit_df,
        ETA_BASE_FACTORIES,
        target_column='trip_duration_minutes',
        n_splits=STACKING_N_SPLITS,
    )
)
display(eta_oof_fold_summary.style.set_caption(
    'ETA stacking — chronological OOF folds bên trong fit_df'
))

eta_validation_meta_model = make_eta_meta_model()
eta_validation_meta_model.fit(eta_fit_oof_X, eta_fit_oof_y)

eta_validation_base_models = fit_regressor_dict(
    ETA_BASE_FACTORIES, X_fit_t, y_fit_eta
)
eta_validation_base_predictions = {
    name: np.maximum(np.asarray(model.predict(X_validation_t), dtype=float), 0.0)
    for name, model in eta_validation_base_models.items()
}
eta_validation_meta_X = np.column_stack([
    eta_validation_base_predictions[name] for name in ETA_BASE_ORDER
])
eta_validation_ensemble_pred = np.maximum(
    eta_validation_meta_model.predict(eta_validation_meta_X), 0.0
)

eta_validation_rows = []
for name in ETA_BASE_ORDER:
    eta_validation_rows.append({
        'Model': name,
        'Evaluation rows': len(validation_df),
        **regression_metrics(y_validation_eta, eta_validation_base_predictions[name]),
    })
eta_validation_rows.append({
    'Model': ETA_ENSEMBLE_NAME,
    'Evaluation rows': len(validation_df),
    **regression_metrics(y_validation_eta, eta_validation_ensemble_pred),
})
eta_validation_results_df = pd.DataFrame(eta_validation_rows).sort_values(
    ['MAE', 'RMSE', 'MedianAE']
).reset_index(drop=True)

display(eta_validation_results_df.style.set_caption(
    'ETA validation — 4 trường hợp trên toàn bộ validation như notebook ban đầu'
))

best_eta_model_name = eta_validation_results_df.iloc[0]['Model']
print(f'Selected ETA model from full validation: {best_eta_model_name}')

start_preprocessor = make_start_preprocessor()
X_train_t = to_dense(start_preprocessor.fit_transform(train_df[START_FEATURES]))
X_test_t = to_dense(start_preprocessor.transform(test_df[START_FEATURES]))
feature_names = start_preprocessor.get_feature_names_out()
y_train_eta = train_df['trip_duration_minutes'].to_numpy()
y_test_eta = test_df['trip_duration_minutes'].to_numpy()

eta_train_oof_X, eta_train_oof_y, eta_train_oof_mask, eta_final_oof_fold_summary = (
    chronological_oof_regression(
        train_df,
        ETA_BASE_FACTORIES,
        target_column='trip_duration_minutes',
        n_splits=STACKING_N_SPLITS,
    )
)
display(eta_final_oof_fold_summary.style.set_caption(
    'ETA final stacking — chronological OOF folds trên toàn bộ external train'
))

eta_final_meta_model = make_eta_meta_model()
eta_final_meta_model.fit(eta_train_oof_X, eta_train_oof_y)
eta_final_base_models = fit_regressor_dict(
    ETA_BASE_FACTORIES, X_train_t, y_train_eta
)
eta_ensemble_model = BoostingPredictionRegressor(
    base_models=eta_final_base_models,
    meta_model=eta_final_meta_model,
    model_order=ETA_BASE_ORDER,
)

eta_models_all = {
    **eta_final_base_models,
    ETA_ENSEMBLE_NAME: eta_ensemble_model,
}
eta_test_predictions = {
    name: np.maximum(np.asarray(model.predict(X_test_t), dtype=float), 0.0)
    for name, model in eta_models_all.items()
}
eta_test_results_df = pd.DataFrame([
    {'Model': name, **regression_metrics(y_test_eta, eta_test_predictions[name])}
    for name in [*ETA_BASE_ORDER, ETA_ENSEMBLE_NAME]
]).sort_values(['MAE', 'RMSE', 'MedianAE']).reset_index(drop=True)

display(eta_test_results_df.style.set_caption(
    'ETA final test — bảng kết quả 4 trường hợp'
))

final_eta_model = eta_models_all[best_eta_model_name]
final_eta_model_matrix_kind = 'dense'
best_eta_pred = eta_test_predictions[best_eta_model_name]

print(f'Transformed full train shape: {X_train_t.shape}')
print(f'Transformed final test shape: {X_test_t.shape}')
print(f'Deployment ETA model selected only from full validation: {best_eta_model_name}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=eta_validation_results_df, x='MAE', y='Model', ax=axes[0], color='C0')
axes[0].set_title('ETA validation — 4 trường hợp')
axes[0].set_xlabel('MAE (phút)')

sns.barplot(data=eta_test_results_df, x='MAE', y='Model', ax=axes[1], color='C0')
axes[1].set_title('ETA final test — 4 trường hợp')
axes[1].set_xlabel('MAE (phút)')

plt.tight_layout()
plt.show()

In [ ]:
plot_df = pd.DataFrame({
    'Actual': np.asarray(y_test_eta),
    'Predicted': np.asarray(best_eta_pred),
}).sample(min(2500, len(test_df)), random_state=RANDOM_STATE)

plt.figure(figsize=(7, 6))
sns.scatterplot(data=plot_df, x='Actual', y='Predicted', alpha=.5, color='C0')
max_value = max(plot_df['Actual'].max(), plot_df['Predicted'].max())
plt.plot([0, max_value], [0, max_value], linestyle='--')
plt.title(f'Final test — actual vs predicted ({best_eta_model_name})')
plt.xlabel('Actual full elapsed duration (phút)')
plt.ylabel('Predicted duration (phút)')
plt.tight_layout()
plt.show()

## 10. Delay Magnitude Model


Delay magnitude is derived directly from the same ETA/duration prediction:

\[
\widehat{Delay} = \widehat{Duration} - OSRMTime
\]

This keeps the predicted ETA and predicted delay always consistent. It is a **proxy delay relative to the OSRM routing baseline**, not an official SLA delay.


In [ ]:
y_test_delay = test_df['delay_minutes']

delay_pred = best_eta_pred - test_df['osrm_time'].to_numpy()

delay_results_df = pd.DataFrame([
    {
        'Model': 'Zero-delay / OSRM baseline',
        **regression_metrics(y_test_delay, np.zeros(len(y_test_delay)))
    },
    {
        'Model': f'Derived from {best_eta_model_name}',
        **regression_metrics(y_test_delay, delay_pred)
    },
]).sort_values('MAE').reset_index(drop=True)

display(delay_results_df)

In [ ]:
delay_plot = pd.DataFrame({
    'Actual delay': y_test_delay.to_numpy(),
    'Predicted delay': delay_pred,
}).sample(min(2500, len(test_df)), random_state=RANDOM_STATE)

plt.figure(figsize=(7, 6))
sns.scatterplot(data=delay_plot, x='Actual delay', y='Predicted delay', alpha=.5, color='C0')
low = min(delay_plot.min())
high = max(delay_plot.max())
plt.plot([low, high], [low, high], linestyle='--')
plt.title('Delay magnitude: actual vs predicted')
plt.xlabel('Actual delay (phút)')
plt.ylabel('Predicted delay (phút)')
plt.tight_layout()
plt.show()

## 11. Severe Delay Classification — 3 Base Models + Boosting Ensemble


Three classifiers are trained independently: **Extra Trees Classifier**, **Rotation Forest Classifier**, and **Bagging Classifier**.

As with the ETA section, the full 20% `validation_df` is kept intact to evaluate the four candidates. Inside the 80% `fit_df`, `TimeSeriesSplit` generates out-of-fold probabilities for the three classifiers. These probabilities are used to:

- train a `GradientBoostingClassifier`;
- choose a probability threshold for each base model and for the ensemble;
- no validation row is used to fit the meta-model or choose a threshold.

As a result, all four classifiers are compared on the same complete validation set, matching the original notebook's protocol.


In [ ]:
print(f'Severe delay label: delay_minutes > {DELAY_THRESHOLD_MINUTES} phút')
print('Evaluation scope: 3 classifier riêng lẻ + 1 boosting ensemble.')

In [ ]:
def classification_metrics_at_threshold(y_true, probability, threshold):
    pred = (np.asarray(probability) >= threshold).astype(int)
    return {
        'Accuracy': accuracy_score(y_true, pred),
        'Precision': precision_score(y_true, pred, zero_division=0),
        'Recall': recall_score(y_true, pred, zero_division=0),
        'F1': f1_score(y_true, pred, zero_division=0),
        'ROC_AUC': safe_roc_auc(y_true, probability),
        'PR_AUC': average_precision_score(y_true, probability),
    }


def optimize_probability_threshold(y_true, probability):
    rows = []
    for threshold in np.linspace(0.05, 0.95, 91):
        rows.append({
            'Threshold': threshold,
            **classification_metrics_at_threshold(y_true, probability, threshold),
        })
    table = pd.DataFrame(rows)
    best = table.sort_values(['F1', 'Recall', 'Precision'], ascending=False).iloc[0]
    return float(best['Threshold']), table


def fit_classifier_dict(factories, X, y):
    models = {}
    balanced_flags = {}
    for position, (name, factory) in enumerate(factories.items(), start=1):
        start = time.perf_counter()
        model, balanced_used = fit_classifier_with_balancing(factory(), X, y)
        models[name] = model
        balanced_flags[name] = balanced_used
        print(
            f'[{position}/{len(factories)}] fitted {name}: '
            f'{time.perf_counter() - start:.2f}s; balanced_weight={balanced_used}'
        )
        gc.collect()
    return models, balanced_flags


def chronological_oof_classification(raw_data, factories, y, n_splits=5):
    """Tạo xác suất severe-delay OOF theo thời gian, fit preprocessor theo từng fold."""
    raw_data = raw_data.sort_values('trip_creation_time').reset_index(drop=True)
    y = np.asarray(y, dtype=int)
    model_names = list(factories)
    oof_probabilities = np.full((len(raw_data), len(model_names)), np.nan, dtype=float)
    fold_rows = []
    splitter = TimeSeriesSplit(n_splits=n_splits)

    for fold, (train_idx, holdout_idx) in enumerate(splitter.split(raw_data), start=1):
        fold_start = time.perf_counter()
        y_fold_train = y[train_idx]
        print(
            f'[Classification OOF fold {fold}/{n_splits}] '
            f'train={len(train_idx):,}, holdout={len(holdout_idx):,}, '
            f'positive_rate={y_fold_train.mean():.2%}'
        )

        if np.unique(y_fold_train).size < 2:
            constant_probability = float(y_fold_train.mean())
            oof_probabilities[holdout_idx, :] = constant_probability
            print(f'  - single-class fold, dùng probability hằng={constant_probability:.4f}')
            fold_rows.append({
                'Fold': fold,
                'Train rows': len(train_idx),
                'OOF rows': len(holdout_idx),
                'Train positive rate': y_fold_train.mean(),
                'OOF positive rate': y[holdout_idx].mean(),
                'Seconds': time.perf_counter() - fold_start,
            })
            continue

        fold_preprocessor = make_start_preprocessor()
        X_fold_train = to_dense(
            fold_preprocessor.fit_transform(raw_data.iloc[train_idx][START_FEATURES])
        )
        X_fold_holdout = to_dense(
            fold_preprocessor.transform(raw_data.iloc[holdout_idx][START_FEATURES])
        )

        for model_position, (name, factory) in enumerate(factories.items()):
            model_start = time.perf_counter()
            model, balanced_used = fit_classifier_with_balancing(
                factory(), X_fold_train, y_fold_train
            )
            oof_probabilities[holdout_idx, model_position] = (
                positive_class_probability(model, X_fold_holdout)
            )
            print(
                f'  - {name}: {time.perf_counter() - model_start:.2f}s; '
                f'balanced_weight={balanced_used}'
            )
            del model
            gc.collect()

        fold_rows.append({
            'Fold': fold,
            'Train rows': len(train_idx),
            'OOF rows': len(holdout_idx),
            'Train positive rate': y_fold_train.mean(),
            'OOF positive rate': y[holdout_idx].mean(),
            'Seconds': time.perf_counter() - fold_start,
        })
        del fold_preprocessor, X_fold_train, X_fold_holdout
        gc.collect()

    valid_mask = np.isfinite(oof_probabilities).all(axis=1)
    if not valid_mask.any():
        raise RuntimeError('Không tạo được probability OOF cho severe-delay ensemble.')
    if np.unique(y[valid_mask]).size < 2:
        raise RuntimeError('OOF rows chỉ chứa một class; không thể fit boosting classifier.')

    return (
        oof_probabilities[valid_mask],
        y[valid_mask],
        valid_mask,
        pd.DataFrame(fold_rows),
    )


y_fit_cls = (fit_df['delay_minutes'] > DELAY_THRESHOLD_MINUTES).astype(int).to_numpy()
y_validation_cls = (
    validation_df['delay_minutes'] > DELAY_THRESHOLD_MINUTES
).astype(int).to_numpy()

(
    classifier_fit_oof_X,
    classifier_fit_oof_y,
    classifier_fit_oof_mask,
    classifier_oof_fold_summary,
) = chronological_oof_classification(
    fit_df,
    CLASSIFIER_BASE_FACTORIES,
    y_fit_cls,
    n_splits=STACKING_N_SPLITS,
)
display(classifier_oof_fold_summary.style.set_caption(
    'Severe-delay stacking — chronological OOF folds bên trong fit_df'
))

classifier_validation_meta_model = make_classifier_meta_model()
meta_sample_weight = compute_sample_weight(
    class_weight='balanced', y=classifier_fit_oof_y
)
classifier_validation_meta_model.fit(
    classifier_fit_oof_X,
    classifier_fit_oof_y,
    sample_weight=meta_sample_weight,
)
classifier_fit_oof_ensemble_probability = positive_class_probability(
    classifier_validation_meta_model, classifier_fit_oof_X
)

validation_thresholds = {}
classifier_threshold_tables = {}
for position, name in enumerate(CLASSIFIER_BASE_ORDER):
    threshold, threshold_table = optimize_probability_threshold(
        classifier_fit_oof_y, classifier_fit_oof_X[:, position]
    )
    validation_thresholds[name] = threshold
    classifier_threshold_tables[name] = threshold_table

ensemble_threshold, ensemble_threshold_table = optimize_probability_threshold(
    classifier_fit_oof_y, classifier_fit_oof_ensemble_probability
)
validation_thresholds[CLASSIFIER_ENSEMBLE_NAME] = ensemble_threshold
classifier_threshold_tables[CLASSIFIER_ENSEMBLE_NAME] = ensemble_threshold_table

classifier_validation_base_models, classifier_validation_balanced = fit_classifier_dict(
    CLASSIFIER_BASE_FACTORIES, X_fit_t, y_fit_cls
)
classifier_validation_base_probabilities = {
    name: positive_class_probability(model, X_validation_t)
    for name, model in classifier_validation_base_models.items()
}
classifier_validation_meta_X = np.column_stack([
    classifier_validation_base_probabilities[name]
    for name in CLASSIFIER_BASE_ORDER
])
classifier_validation_ensemble_probability = positive_class_probability(
    classifier_validation_meta_model, classifier_validation_meta_X
)

classifier_validation_rows = []
for name in CLASSIFIER_BASE_ORDER:
    classifier_validation_rows.append({
        'Model': name,
        'ProbabilityThreshold': validation_thresholds[name],
        'Evaluation rows': len(validation_df),
        **classification_metrics_at_threshold(
            y_validation_cls,
            classifier_validation_base_probabilities[name],
            validation_thresholds[name],
        ),
    })
classifier_validation_rows.append({
    'Model': CLASSIFIER_ENSEMBLE_NAME,
    'ProbabilityThreshold': validation_thresholds[CLASSIFIER_ENSEMBLE_NAME],
    'Evaluation rows': len(validation_df),
    **classification_metrics_at_threshold(
        y_validation_cls,
        classifier_validation_ensemble_probability,
        validation_thresholds[CLASSIFIER_ENSEMBLE_NAME],
    ),
})

classifier_validation_results_df = pd.DataFrame(classifier_validation_rows).sort_values(
    ['F1', 'ROC_AUC', 'Recall', 'Precision'], ascending=False
).reset_index(drop=True)
display(classifier_validation_results_df.style.set_caption(
    'Severe-delay validation — 4 trường hợp trên toàn bộ validation như notebook ban đầu'
))

best_classifier_name = classifier_validation_results_df.iloc[0]['Model']
print(f'Selected classifier from full validation: {best_classifier_name}')

y_train_cls = (train_df['delay_minutes'] > DELAY_THRESHOLD_MINUTES).astype(int).to_numpy()
y_test_cls = (test_df['delay_minutes'] > DELAY_THRESHOLD_MINUTES).astype(int).to_numpy()

(
    classifier_train_oof_X,
    classifier_train_oof_y,
    classifier_train_oof_mask,
    classifier_final_oof_fold_summary,
) = chronological_oof_classification(
    train_df,
    CLASSIFIER_BASE_FACTORIES,
    y_train_cls,
    n_splits=STACKING_N_SPLITS,
)
display(classifier_final_oof_fold_summary.style.set_caption(
    'Severe-delay final stacking — chronological OOF folds trên toàn bộ external train'
))

classifier_final_meta_model = make_classifier_meta_model()
full_meta_weight = compute_sample_weight(
    class_weight='balanced', y=classifier_train_oof_y
)
classifier_final_meta_model.fit(
    classifier_train_oof_X,
    classifier_train_oof_y,
    sample_weight=full_meta_weight,
)

classifier_final_base_models, classifier_final_balanced = fit_classifier_dict(
    CLASSIFIER_BASE_FACTORIES, X_train_t, y_train_cls
)
classifier_ensemble_model = BoostingPredictionClassifier(
    base_models=classifier_final_base_models,
    meta_model=classifier_final_meta_model,
    model_order=CLASSIFIER_BASE_ORDER,
)

classifier_models_all = {
    **classifier_final_base_models,
    CLASSIFIER_ENSEMBLE_NAME: classifier_ensemble_model,
}
classifier_test_probabilities = {
    name: positive_class_probability(model, X_test_t)
    for name, model in classifier_models_all.items()
}

final_thresholds = validation_thresholds.copy()
classification_test_results_df = pd.DataFrame([
    {
        'Model': name,
        'ProbabilityThreshold': final_thresholds[name],
        'BalancedWeightUsed': (
            classifier_final_balanced.get(name, True)
            if name in CLASSIFIER_BASE_ORDER else True
        ),
        **classification_metrics_at_threshold(
            y_test_cls,
            classifier_test_probabilities[name],
            final_thresholds[name],
        ),
    }
    for name in [*CLASSIFIER_BASE_ORDER, CLASSIFIER_ENSEMBLE_NAME]
]).sort_values(['F1', 'ROC_AUC', 'Recall', 'Precision'], ascending=False).reset_index(drop=True)

display(classification_test_results_df.style.set_caption(
    'Severe-delay final test — bảng kết quả 4 trường hợp'
))

classifier = classifier_models_all[best_classifier_name]
classifier_matrix_kind = 'dense'
best_classification_threshold = final_thresholds[best_classifier_name]
severe_delay_probability = classifier_test_probabilities[best_classifier_name]
severe_delay_pred = (
    severe_delay_probability >= best_classification_threshold
).astype(int)
classification_metrics = classification_test_results_df.query(
    'Model == @best_classifier_name'
).reset_index(drop=True)

print(f'Train positive rate: {y_train_cls.mean():.2%}')
print(f'Test positive rate: {y_test_cls.mean():.2%}')
print(f'Deployment classifier selected only from full validation: {best_classifier_name}')
print(f'Deployment probability threshold: {best_classification_threshold:.2f}')
print(classification_report(y_test_cls, severe_delay_pred, digits=3, zero_division=0))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    data=classifier_validation_results_df,
    x='F1',
    y='Model',
    ax=axes[0],
    color='C0',
)
axes[0].set_title('Severe-delay validation F1 — 4 trường hợp')
axes[0].set_xlim(0, 1)

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test_cls, severe_delay_pred),
    display_labels=['Không severe', 'Severe delay']
).plot(ax=axes[1], values_format='d', colorbar=False)
axes[1].set_title(
    f'Final test — selected: {best_classifier_name}\n'
    f'delay>{DELAY_THRESHOLD_MINUTES} phút, p-threshold={best_classification_threshold:.2f}'
)

plt.tight_layout()
plt.show()

## 12. Delay Propagation — Extra Trees Regressor Only


The start-of-trip model uses only planned information. To represent delay propagation, the notebook creates a snapshot after each observed checkpoint.

At checkpoint \(k\):

\[
StageDelay_k = SegmentActualTime_k - SegmentOSRMTime_k
\]

\[
CumulativeDelay_k = \sum_{i=1}^{k} StageDelay_i
\]

Checkpoints are sorted by `cutoff_timestamp`. `total_stages` is excluded because it is future information. The final score is computed at 25%, 50%, 75%, and 100% progress, with each trip contributing a single observation. **This section only trains an Extra Trees Regressor; no other propagation model is used.**


In [ ]:
stage_df = df.query('segment_actual_time >= 0 and segment_osrm_time > 0').copy()
stage_df = stage_df.sort_values(
    ['trip_uuid', 'od_start_time', 'cutoff_timestamp', '_row_order'],
    na_position='last',
).reset_index(drop=True)

trip_targets = trip_df[[
    'trip_uuid', 'trip_duration_minutes', 'osrm_time', 'osrm_distance', 'delay_minutes'
]].rename(columns={
    'trip_duration_minutes': 'final_trip_duration',
    'osrm_time': 'trip_osrm_time',
    'osrm_distance': 'trip_osrm_distance',
    'delay_minutes': 'final_delay_minutes',
})

stage_df = stage_df.merge(trip_targets, on='trip_uuid', how='inner')
stage_group = stage_df.groupby('trip_uuid', sort=False)

stage_df['stage_index'] = stage_group.cumcount() + 1
stage_df['total_segment_osrm_time'] = stage_group['segment_osrm_time'].transform('sum')
stage_df['cum_actual_time'] = stage_group['segment_actual_time'].cumsum()
stage_df['cum_osrm_time'] = stage_group['segment_osrm_time'].cumsum()
stage_df['current_stage_delay'] = stage_df['segment_actual_time'] - stage_df['segment_osrm_time']
stage_df['cum_stage_delay'] = stage_df['cum_actual_time'] - stage_df['cum_osrm_time']
stage_df['previous_stage_delay'] = stage_group['current_stage_delay'].shift(1).fillna(0)
stage_df['progress_ratio'] = (
    stage_df['cum_osrm_time'] /
    stage_df['total_segment_osrm_time'].replace(0, np.nan)
).clip(0, 1)
stage_df['remaining_segment_plan_time'] = (
    stage_df['total_segment_osrm_time'] - stage_df['cum_osrm_time']
).clip(lower=0)
stage_df['start_hour'] = stage_df['od_start_time'].dt.hour
stage_df['start_dayofweek'] = stage_df['od_start_time'].dt.dayofweek

STAGE_NUMERIC_FEATURES = [
    'segment_actual_time', 'segment_osrm_time', 'segment_osrm_distance',
    'current_stage_delay', 'previous_stage_delay',
    'cum_actual_time', 'cum_osrm_time', 'cum_stage_delay',
    'progress_ratio', 'remaining_segment_plan_time',
    'stage_index',
    'trip_osrm_time', 'trip_osrm_distance',
    'start_hour', 'start_dayofweek',
]
STAGE_CATEGORICAL_FEATURES = ['route_type', 'source_state', 'destination_state']
STAGE_FEATURES = STAGE_NUMERIC_FEATURES + STAGE_CATEGORICAL_FEATURES

assert 'total_stages' not in STAGE_FEATURES
assert stage_df['progress_ratio'].between(0, 1).all()

print(f'Stage-level shape after quality filters: {stage_df.shape}')
print(f'Unique stage trips: {stage_df.trip_uuid.nunique():,}')
print(f'Rows excluded by stage quality rules: {len(df) - len(stage_df):,}')
display(stage_df[STAGE_FEATURES + ['final_delay_minutes']].head())

In [ ]:
stage_train = stage_df.query("data == 'training'").copy().reset_index(drop=True)
stage_test = stage_df.query("data == 'test'").copy().reset_index(drop=True)

stage_overlap = set(stage_train['trip_uuid']).intersection(stage_test['trip_uuid'])
assert len(stage_overlap) == 0, 'Có trip_uuid trùng stage train/test.'

stage_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), STAGE_NUMERIC_FEATURES),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(
                handle_unknown='ignore',
                min_frequency=20,
                sparse_output=True,
            )),
        ]), STAGE_CATEGORICAL_FEATURES),
    ]
)

X_stage_train_t = stage_preprocessor.fit_transform(stage_train[STAGE_FEATURES])
X_stage_test_t = stage_preprocessor.transform(stage_test[STAGE_FEATURES])
stage_feature_names = stage_preprocessor.get_feature_names_out()

propagation_model_name = 'Extra Trees Regressor'
propagation_model = ExtraTreesRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    max_features=1.0,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
)

train_stage_count = stage_train.groupby('trip_uuid')['trip_uuid'].transform('size')
stage_sample_weight = 1.0 / train_stage_count
stage_sample_weight = stage_sample_weight / stage_sample_weight.mean()

propagation_model.fit(
    X_stage_train_t,
    stage_train['final_delay_minutes'],
    sample_weight=stage_sample_weight,
)
propagation_pred = propagation_model.predict(X_stage_test_t)
propagation_pred_series = pd.Series(
    propagation_pred, index=stage_test.index, name='prediction'
)

print(f'Propagation model: {propagation_model_name} (only)')
print(f'Stage train rows: {len(stage_train):,}; trips: {stage_train.trip_uuid.nunique():,}')
print(f'Stage test rows: {len(stage_test):,}; trips: {stage_test.trip_uuid.nunique():,}')
print(f'Stage trip overlap train/test: {len(stage_overlap)}')

In [ ]:
def select_one_checkpoint_per_trip(frame, target_progress):
    """Chọn checkpoint gần target progress nhất; mỗi trip đúng một dòng."""
    distance = (frame['progress_ratio'] - target_progress).abs()
    selected_index = distance.groupby(frame['trip_uuid']).idxmin()
    return selected_index

checkpoint_results = []
checkpoint_predictions = {}
for target_progress, label in [
    (0.25, '25%'),
    (0.50, '50%'),
    (0.75, '75%'),
    (1.00, '100%'),
]:
    selected_index = select_one_checkpoint_per_trip(stage_test, target_progress)
    selected = stage_test.loc[selected_index]
    selected_pred = propagation_pred_series.loc[selected_index].to_numpy()
    checkpoint_predictions[label] = (selected, selected_pred)

    checkpoint_results.append({
        'Checkpoint': label,
        'Trips': selected['trip_uuid'].nunique(),
        'Mean actual progress': selected['progress_ratio'].mean(),
        **regression_metrics(selected['final_delay_minutes'], selected_pred),
    })

checkpoint_results_df = pd.DataFrame(checkpoint_results)
display(checkpoint_results_df)

assert checkpoint_results_df['Trips'].max() <= stage_test['trip_uuid'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=checkpoint_results_df, x='Checkpoint', y='MAE', marker='o', ax=axes[0], color='C0')
axes[0].set_title('Propagation MAE — mỗi trip một checkpoint')
axes[0].set_ylabel('MAE (phút)')

sns.lineplot(data=checkpoint_results_df, x='Checkpoint', y='R2', marker='o', ax=axes[1], color='C0')
axes[1].set_title('Propagation R² — mỗi trip một checkpoint')
axes[1].set_ylabel('R²')

plt.tight_layout()
plt.show()

## 13. Explainability with LIME/SHAP — TEMPORARILY COMMENTED OUT


All of LIME/SHAP remains commented out so the notebook can focus on benchmarking exactly the 4 candidates for ETA/classification and a single Extra Trees model for propagation.

Once the run is complete, the code can be uncommented to explain only the selected `final_eta_model`, `classifier`, and `propagation_model`.


## 14. Single-Shipment Prediction Example


In [ ]:
def predict_trip_at_start(row):
    """Dự đoán duration, ETA, proxy delay và severe-delay probability cho một trip."""
    X_row = row[START_FEATURES].to_frame().T
    X_row_t = to_dense(start_preprocessor.transform(X_row))

    predicted_duration = max(float(final_eta_model.predict(X_row_t)[0]), 0.0)
    predicted_delay = predicted_duration - float(row['osrm_time'])
    severe_probability = float(positive_class_probability(
        classifier, X_row_t
    )[0])
    predicted_eta = row['trip_start_time'] + pd.to_timedelta(
        predicted_duration, unit='m'
    )

    return pd.Series({
        'trip_uuid': row['trip_uuid'],
        'origin': row['source_name'],
        'destination': row['destination_name'],
        'selected ETA model': best_eta_model_name,
        'selected severe-delay classifier': best_classifier_name,
        'OSRM baseline minutes': row['osrm_time'],
        'Predicted full elapsed minutes': predicted_duration,
        'Predicted proxy delay minutes': predicted_delay,
        'Predicted ETA': predicted_eta,
        'Severe delay probability': severe_probability,
        'Classification probability threshold': best_classification_threshold,
        'Actual full elapsed minutes': row['trip_duration_minutes'],
        'Actual proxy delay minutes': row['delay_minutes'],
        'Actual ETA': row['actual_eta'],
    })

example_prediction = predict_trip_at_start(test_df.iloc[0])
display(example_prediction.to_frame('value'))

## 15. Model Export and Conclusion


In [ ]:
SAVE_ARTIFACTS = True

if SAVE_ARTIFACTS:
    artifact_dir = Path('/kaggle/working/artifacts') if Path('/kaggle/working').exists() else Path('artifacts')
    artifact_dir.mkdir(exist_ok=True, parents=True)

    joblib.dump(start_preprocessor, artifact_dir / 'start_preprocessor.joblib')
    joblib.dump(final_eta_model, artifact_dir / 'eta_model_selected.joblib')
    joblib.dump(eta_models_all, artifact_dir / 'eta_models_all_4_cases.joblib')
    joblib.dump(classifier, artifact_dir / 'severe_delay_classifier_selected.joblib')
    joblib.dump(classifier_models_all, artifact_dir / 'severe_delay_models_all_4_cases.joblib')
    joblib.dump(stage_preprocessor, artifact_dir / 'stage_preprocessor.joblib')
    joblib.dump(propagation_model, artifact_dir / 'delay_propagation_extra_trees.joblib')

    eta_validation_results_df.to_csv(
        artifact_dir / 'eta_validation_4_cases.csv', index=False
    )
    eta_test_results_df.to_csv(
        artifact_dir / 'eta_final_test_4_cases.csv', index=False
    )
    classifier_validation_results_df.to_csv(
        artifact_dir / 'severe_delay_validation_4_cases.csv', index=False
    )
    classification_test_results_df.to_csv(
        artifact_dir / 'severe_delay_final_test_4_cases.csv', index=False
    )
    checkpoint_results_df.to_csv(
        artifact_dir / 'delay_propagation_extra_trees_checkpoints.csv', index=False
    )

    metadata = {
        'notebook_version': 'Delhivery-Extra',
        'ground_truth_duration': 'trip_end_time - trip_start_time',
        'delay_definition': 'trip_duration_minutes - osrm_time',
        'delay_threshold_minutes': DELAY_THRESHOLD_MINUTES,
        'classification_probability_threshold': best_classification_threshold,
        'start_features': START_FEATURES,
        'stage_features': STAGE_FEATURES,
        'eta_cases': [*ETA_BASE_ORDER, ETA_ENSEMBLE_NAME],
        'classification_cases': [*CLASSIFIER_BASE_ORDER, CLASSIFIER_ENSEMBLE_NAME],
        'ensemble_method': 'three out-of-sample base predictions/probabilities -> GradientBoosting meta-model',
        'selected_eta_model_from_validation': str(best_eta_model_name),
        'selected_classifier_from_validation': str(best_classifier_name),
        'propagation_model': propagation_model_name,
        'lime_shap_enabled': False,
        'external_train_trips': int(len(train_df)),
        'final_test_trips': int(len(test_df)),
    }
    (artifact_dir / 'metadata.json').write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8'
    )
    print(f'Đã lưu artifacts tại: {artifact_dir.resolve()}')
else:
    print('SAVE_ARTIFACTS=False — đổi thành True để lưu model vào /kaggle/working/artifacts.')

### Conclusion — Delhivery-Extra-v1

- ETA Prediction runs only **Extra Trees Regressor**, **Rotation Forest Regressor**, **Bagging Regressor**, and **Boosting Ensemble**.
- Severe Delay Classification runs only **Extra Trees Classifier**, **Rotation Forest Classifier**, **Bagging Classifier**, and **Boosting Ensemble**.
- The main split matches the original notebook: the first 80% of external training is fit, the final 20% is validation, with no shuffling.
- All four candidates are evaluated on the complete validation set; validation is no longer split into meta-train and meta-evaluation.
- The boosting meta-model and classification thresholds are learned from chronological out-of-fold predictions inside the fit set.
- Before the final test, the meta-model is refit using OOF predictions from the full external train set; the three base models are refit on the full external train set.
- Delay Propagation uses only an **Extra Trees Regressor**, evaluated at the 25%, 50%, 75%, and 100% checkpoints.
- The final deployed model is still chosen using validation, not the final test set.
- LIME/SHAP remain commented out to avoid a long runtime outside the required scope.
